# AML Synthetic Data Generator

### 1. Imports and Configuration

In [1]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

rng = np.random.default_rng(42)

### 2. Municipalities

In [2]:
CO_MUNICIPALITIES = [
    "Bogotá", "Medellín", "Cali", "Barranquilla", "Cartagena",
    "Cúcuta", "Bucaramanga", "Pereira", "Santa Marta", "Ibagué",
    "Manizales", "Villavicencio", "Pasto", "Montería", "Neiva",
    "Armenia", "Sincelejo", "Valledupar", "Tunja", "Popayán"
]

### 3. Product Catalog

These products behave differently in terms of:

- Transaction frequency
- Average ticket amount
- Debit/Credit ratio

In [3]:
PRODUCTS = {
    "Cuenta de ahorros": {
        "weight": 0.95,
        "avg_tx_multiplier": 1.0,
        "amount_multiplier": 1.0,
        "debit_prob": 0.60
    },
    "CDT": {
        "weight": 0.25,
        "avg_tx_multiplier": 0.10,
        "amount_multiplier": 8.0,
        "debit_prob": 0.20
    },
    "Crédito libre inversión": {
        "weight": 0.35,
        "avg_tx_multiplier": 0.35,
        "amount_multiplier": 2.5,
        "debit_prob": 0.75
    },
    "Crédito libranza": {
        "weight": 0.30,
        "avg_tx_multiplier": 0.25,
        "amount_multiplier": 1.8,
        "debit_prob": 0.70
    }
}

### 4. Customer Generator

In [4]:
def generate_customers(n_customers: int = 5000) -> pd.DataFrame:

    customer_ids = np.arange(1, n_customers + 1)

    municipalities = rng.choice(CO_MUNICIPALITIES, size=n_customers)

    risk_rating = rng.choice(
        ["low", "medium", "high"],
        size=n_customers,
        p=[0.70, 0.25, 0.05]
    )

    today = datetime(2025, 1, 1)

    onboarding_days_ago = rng.integers(30, 365 * 10, size=n_customers)
    onboarding_date = [today - timedelta(days=int(d)) for d in onboarding_days_ago]

    age = rng.integers(18, 80, size=n_customers)

    assets = rng.lognormal(mean=11, sigma=1.0, size=n_customers)
    liabilities = assets * rng.uniform(0.1, 0.9, size=n_customers)

    monthly_income = rng.lognormal(mean=9, sigma=0.7, size=n_customers)
    monthly_expenses = monthly_income * rng.uniform(0.4, 1.1, size=n_customers)

    pep_flag = rng.choice([0, 1], size=n_customers, p=[0.98, 0.02])

    customers = pd.DataFrame({
        "customer_id": customer_ids,
        "municipality": municipalities,
        "risk_rating": risk_rating,
        "onboarding_date": onboarding_date,
        "age": age,
        "assets_total": assets,
        "liabilities_total": liabilities,
        "monthly_income": monthly_income,
        "monthly_expenses": monthly_expenses,
        "pep_flag": pep_flag
    })

    return customers

### 5. Customer Products Table

This creates a realistic many-to-one relationship:

- One customer can own multiple products.
- One product can generate many transactions.

In [5]:
def generate_customer_products(customers: pd.DataFrame) -> pd.DataFrame:

    product_rows = []
    product_id = 1

    for _, cust in customers.iterrows():

        for product_name, config in PRODUCTS.items():

            has_product = rng.random() < config["weight"]

            if has_product:

                open_days_ago = rng.integers(1, 365 * 8)
                open_date = datetime(2025, 1, 1) - timedelta(days=int(open_days_ago))

                product_rows.append({
                    "product_id": product_id,
                    "customer_id": cust["customer_id"],
                    "product_type": product_name,
                    "open_date": open_date,
                    "product_status": "active"
                })

                product_id += 1

    return pd.DataFrame(product_rows)

### 6. Transaction Volume Logic

Different products generate different transaction volumes.

In [6]:
def _sample_tx_count(risk_rating: str, product_type: str) -> int:

    base = 90

    risk_mult = {
        "low": 0.8,
        "medium": 1.0,
        "high": 1.5
    }[risk_rating]

    product_mult = PRODUCTS[product_type]["avg_tx_multiplier"]

    return int(rng.poisson(lam=base * risk_mult * product_mult))

### 7. Realistic Transaction Generator

- Product-level behavior.
- Merchant/channel simulation.
- Realistic transaction descriptions.
- Realistic amounts.
- Business hour patterns.
- Salary cycles.

In [7]:
TRANSACTION_CHANNELS = [
    "ATM",
    "Branch",
    "Mobile App",
    "Web",
    "PSE",
    "POS"
]

TRANSACTION_DESCRIPTIONS = {
    "Cuenta de ahorros": [
        "Transferencia",
        "Pago servicios",
        "Compra comercio",
        "Retiro cajero",
        "Pago nómina",
        "Consignación"
    ],

    "CDT": [
        "Constitución CDT",
        "Renovación CDT",
        "Pago intereses CDT",
        "Cancelación CDT"
    ],

    "Crédito libre inversión": [
        "Desembolso crédito",
        "Pago cuota crédito",
        "Abono extraordinario"
    ],

    "Crédito libranza": [
        "Desembolso libranza",
        "Descuento nómina",
        "Pago cuota libranza"
    ]
}


def generate_transactions(
    customers: pd.DataFrame,
    customer_products: pd.DataFrame,
    start_date: datetime,
    end_date: datetime,
    suspicious_ratio: float = 0.03
) -> pd.DataFrame:

    tx_rows = []
    tx_id = 1

    n_days = (end_date - start_date).days

    customer_lookup = customers.set_index("customer_id")

    for _, product in customer_products.iterrows():

        cust = customer_lookup.loc[product["customer_id"]]

        product_type = product["product_type"]

        total_tx = _sample_tx_count(
            cust["risk_rating"],
            product_type
        )

        for _ in range(total_tx):

            day_offset = int(rng.integers(0, n_days))
            tx_date = start_date + timedelta(days=day_offset)

            # More realistic business hours
            tx_time = tx_date + timedelta(
                hours=int(np.clip(rng.normal(13, 4), 0, 23)),
                minutes=int(rng.integers(0, 60)),
                seconds=int(rng.integers(0, 60))
            )

            income_factor = max(cust["monthly_income"], 1)

            amount_multiplier = PRODUCTS[product_type]["amount_multiplier"]

            base_amount = (income_factor / 30) * amount_multiplier

            amount = rng.lognormal(
                mean=np.log(base_amount),
                sigma=0.9
            )

            amount = float(np.clip(amount, 5_000, 150_000_000))

            debit_probability = PRODUCTS[product_type]["debit_prob"]

            tx_type = rng.choice(
                ["debit", "credit"],
                p=[debit_probability, 1 - debit_probability]
            )

            description = rng.choice(
                TRANSACTION_DESCRIPTIONS[product_type]
            )

            channel = rng.choice(
                TRANSACTION_CHANNELS,
                p=[0.15, 0.05, 0.35, 0.20, 0.15, 0.10]
            )

            tx_rows.append({
                "tx_id": tx_id,
                "customer_id": cust.name,
                "product_id": product["product_id"],
                "product_type": product_type,
                "tx_timestamp": tx_time,
                "tx_amount": round(amount, 2),
                "tx_type": tx_type,
                "tx_channel": channel,
                "tx_description": description,
                "municipality": cust["municipality"],
                "risk_rating": cust["risk_rating"],
                "is_suspicious": 0,
                "pattern_tag": "normal"
            })

            tx_id += 1

    transactions = pd.DataFrame(tx_rows)

    # ======================================
    # Suspicious Patterns
    # ======================================

    n_suspicious = int(len(transactions) * suspicious_ratio)

    suspicious_idx = rng.choice(
        transactions.index,
        size=n_suspicious,
        replace=False
    )

    # Structuring
    structuring = suspicious_idx[: int(n_suspicious * 0.5)]

    transactions.loc[
        structuring,
        "tx_amount"
    ] = rng.uniform(9_000_000, 9_900_000, size=len(structuring))

    transactions.loc[
        structuring,
        "pattern_tag"
    ] = "structuring"

    transactions.loc[
        structuring,
        "is_suspicious"
    ] = 1

    # Rapid movement
    rapid_flow = suspicious_idx[int(n_suspicious * 0.5):]

    transactions.loc[
        rapid_flow,
        "pattern_tag"
    ] = "rapid_in_out"

    transactions.loc[
        rapid_flow,
        "is_suspicious"
    ] = 1

    return transactions

### 8. Main Execution

In [8]:
if __name__ == "__main__":

    customers = generate_customers(5000)

    customer_products = generate_customer_products(customers)

    start = datetime(2024, 1, 1)
    end = datetime(2024, 12, 31)

    transactions = generate_transactions(
        customers,
        customer_products,
        start,
        end
    )

    print(customers.head())
    print(customer_products.head())
    print(transactions.head())

    print("\nCustomers:", customers.shape)
    print("Products:", customer_products.shape)
    print("Transactions:", transactions.shape)

   customer_id municipality risk_rating onboarding_date  age  assets_total  \
0            1     Medellín         low      2024-03-06   20   9644.530385   
1            2      Armenia         low      2019-12-14   62  31913.511349   
2            3     Montería        high      2019-04-10   31  22262.188712   
3            4  Santa Marta      medium      2017-08-30   62  67375.641780   
4            5  Santa Marta         low      2019-04-02   53  27237.004781   

   liabilities_total  monthly_income  monthly_expenses  pep_flag  
0        4848.527271     5348.343237       4453.541921         0  
1       25856.235061     3575.389803       1502.085224         0  
2        5126.008116     7904.576462       3826.750581         0  
3       16591.101843     2871.716730       2760.109186         0  
4       13887.452465     8846.981965       6561.252283         0  
   product_id  customer_id             product_type  open_date product_status
0           1            1        Cuenta de ahorros

### 9. Export Files

In [9]:
customers.to_excel("synthetic_customers.xlsx", index=False)

customer_products.to_excel("synthetic_customer_products.xlsx", index=False)

transactions.to_excel("synthetic_transactions.xlsx", index=False)